# Train YOLOv11 using Google Colab GPU
**CRITICAL:** Before running any code, open the Colab file explorer (folder icon on the left) and upload `colab_dataset.zip` directly into the default `/content/` folder.

In [ ]:
import torch
import os

if torch.cuda.is_available():
    print(f"NVIDIA GPU (CUDA) is available! Using: {torch.cuda.get_device_name(0)} 🚀")
else:
    print("No GPU detected. Make sure your Colab runtime is set to T4 GPU! 🐢")

In [ ]:
# 1. Unzip the dataset (The '-o' flag overwrites if you run it twice)
!unzip -o colab_dataset.zip

# 2. DEBUG CHECK: Verify the files actually unzipped properly
# Checking for the YAML file in the /content/ folder
if os.path.exists('/content/segmentation/dataset.yaml'):
    print("✅ dataset.yaml found successfully!")
else:
    print("❌ ERROR: dataset.yaml is MISSING. Ensure it was uploaded correctly.")

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

# Load the nano segmentation model
model = YOLO('yolo11n-seg.pt')

results = model.train(
    data='/content/segmentation/dataset.yaml', 
    epochs=50, 
    imgsz=640,
    batch=16,
    device=0,            
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    patience=20
)

In [ ]:
# DEBUG & RESULTS CHECK: Run this after training finishes!
from IPython.display import Image, display
import glob

# 1. Display the training progress graph (shows how accuracy improved over epochs)
results_img = '/content/runs/segment/train/results.png'
if os.path.exists(results_img):
    print("\n--- Training Progress Over Epochs ---")
    display(Image(filename=results_img))

# 2. Test the model on a random image from the test set
print("\n--- Running Test Inference ---")
best_model = YOLO('/content/runs/segment/train/weights/best.pt')

# Grab the first image in the test folder
test_images = glob.glob('/content/segmentation_split/test/images/*.jpg')
if test_images:
    test_img = test_images[0]
    print(f"Testing on: {test_img}")
    # Run prediction and save the annotated image
    prediction = best_model.predict(source=test_img, save=True)
    
    # Find the newly saved prediction image and display it
    pred_dir = prediction[0].save_dir
    pred_img = os.path.join(pred_dir, os.path.basename(test_img))
    display(Image(filename=pred_img))
else:
    print("No test images found.")